In [2]:
import GtoTmodel
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers =10 # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text

In [4]:
# Check if GPU is available and set the device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [5]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim,
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [6]:
from Circuits import Circuits
circuits = Circuits()


Loading dataset files...
Loaded dataset files successfully.


In [7]:
# Define the file path to load the model and hyperparameters
load_path = "model_checkpoint.pth"

# Load the checkpoint
checkpoint = torch.load(load_path)

# Restore the model state and hyperparameters
model.load_state_dict(checkpoint['model_state_dict'])

# Restore hyperparameters if needed
embed_dim = checkpoint['embed_dim']
num_heads = checkpoint['num_heads']
num_layers = checkpoint['num_layers']
dropout = checkpoint['dropout']
text_vocab_size = checkpoint['text_vocab_size']
graph_input_dim = checkpoint['graph_input_dim']

print(f"Model and optimizer state loaded from {load_path}")

Model and optimizer state loaded from model_checkpoint.pth


C:\Users\MSI\AppData\Local\Temp\ipykernel_17484\3870851804.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(load_path)


In [8]:
model.eval()
index=torch.randint(0, len(circuits.graphs), (1,)).item() # Random index to test
graph=torch.tensor(circuits.graphs[index],dtype=torch.float32).to(device)
text=torch.tensor(circuits.component_indices[index]).to(device)
graph.shape,text.shape
graph =F.pad(graph, (0, 310 - graph.size(0) if graph.size(1) < 310 else 0, 0, 0), mode='constant', value=9)
graph.size()
test_graph = graph.view(1,graph.size(0),310).to(torch.float32)
test_text = text.view(1,text.size(0)) 
circuits.get_component_fromlist(test_text[0].tolist())

['VDD',
 'VSS',
 'IIN1',
 'VOUT1',
 'R1',
 'R1_P',
 'R1_N',
 'R2',
 'R2_P',
 'R2_N',
 'R3',
 'R3_P',
 'R3_N',
 'NM1',
 'NM1_D',
 'NM1_G',
 'NM1_S',
 'NM1_B',
 'NM2',
 'NM2_D',
 'NM2_G',
 'NM2_S',
 'NM2_B']

In [9]:


with torch.no_grad(): 
        if test_text.dim() == 1:
            test_text = test_text.unsqueeze(0)

        if test_graph.dim() == 1:
            test_graph = test_graph.unsqueeze(0)

        # Forward pass through the model
        output = model(test_graph, test_text[:, :-1])  # Exclude the last token for input
        # print("Output Shape:", output.shape)

        # Select the prediction for the last timestep
        last_timestep_output = output[:, -1, :]
        predicted_token = torch.argmax(last_timestep_output, dim=-1)
        # Get the top five predictions for the last timestep
        top_five_predictions = torch.topk(last_timestep_output, 5, dim=-1).indices.squeeze(0)

        # Convert the top five predictions to actual tokens
        top_five_tokens = [circuits.get_component(token.item()) for token in top_five_predictions]

        print("Top Five Predicted Tokens:", top_five_tokens)

print("Predicted Token:", circuits.get_component(predicted_token.item()))  # Convert to actual token
print("Actual Next Token:", circuits.get_component(int(test_text[:, -1])))  # Compare with the actual next token

Top Five Predicted Tokens: ['R5', 'TRANSMISSION_GATE2_A', 'NM7_G', 'DIO6_N', 'NM5_G']
Predicted Token: R5
Actual Next Token: NM2_B


In [11]:
graph_data,text_data=circuits.data_lodder()

In [12]:
# Load the saved model checkpoint
checkpoint_path = "model_checkpoint.pth"
checkpoint = torch.load(checkpoint_path)

# Reinitialize the model with the saved hyperparameters
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim=checkpoint['graph_input_dim'],
    text_vocab_size=checkpoint['text_vocab_size'],
    embed_dim=checkpoint['embed_dim'],
    num_heads=checkpoint['num_heads'],
    num_layers=checkpoint['num_layers'],
    dropout=checkpoint['dropout']
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Select a sample graph input
sample_graph = graph_data[0].unsqueeze(0).to(device)  # Add batch dimension

# Generate a sequence
start_token = torch.tensor([0], dtype=torch.long).to(device)  # Assuming 0 is the start token
generated_sequence = [start_token.item()]

for _ in range(5000):  # Generate up to 50 tokens
    input_sequence = torch.tensor(generated_sequence, dtype=torch.long).unsqueeze(0).to(device)
    output = model(sample_graph, input_sequence)
    next_token = torch.argmax(output[:, -1, :], dim=-1).item()  # Get the most probable next token
    generated_sequence.append(next_token)
    if next_token == 892:  # Assuming 892 is the end token
        break

# Convert indices back to components
generated_text = circuits.get_component_fromlist(generated_sequence)

print("Generated Sequence:", generated_text)

C:\Users\MSI\AppData\Local\Temp\ipykernel_17484\179958460.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)
c:\Users\MSI\miniconda

Generated Sequence: ['INVERTER2', 'INVERTER10', 'R3', 'INVERTER6_VDD', 'PM21', 'VBB4', 'INVERTER10_VDD', 'R11_P', 'PNP11_E', 'TRANSMISSION_GATE8', 'PM20_B', 'INVERTER4_VSS', 'R5_N', 'PNP8_C', 'PM7_D', 'VB1', 'NM18_D', 'VIN10', 'NPN8_E', 'PFD1_A', 'C1_P', 'VCLK3', 'PNP11', 'PNP5_C', 'L4', 'L21', 'PM21_G', 'R1_N', 'INVERTER6', 'NM30_D', 'NPN25_C', 'TRANSMISSION_GATE1_VSS', 'TRANSMISSION_GATE12_C', 'C7_N', 'L9', 'TRANSMISSION_GATE2_VDD', 'NM31_S', 'R14_P', 'NM18_S', 'NPN23_E', 'L17', 'R9_P', 'TRANSMISSION_GATE8_VSS', 'NM7_B', 'L11', 'NM34_B', 'VREF2', 'C8_N', 'R16_N', 'PM7_S', 'NPN8_B', 'L10', 'INVERTER6_A', 'NPN19_C', 'L10_N', 'PM18_D', 'R7', 'NM4', 'R22_P', 'NM32_G', 'DIO1', 'NPN25', 'INVERTER4_VDD', 'L6', 'PM11_D', 'PM16_B', 'C6_N', 'DIO2', 'NM12_D', 'TRANSMISSION_GATE6_B', 'NPN22_B', 'NM15_B', 'PM13', 'L14_N', 'C11', 'VB4', 'INVERTER8_VSS', 'INVERTER10_Q', 'R23_P', 'PNP10_C', 'C4', 'NM26_B', 'NM31_D', 'NM1_S', 'PFD1_VDD', 'NM1', 'NM18', 'NM27_B', 'NPN16', 'PNP2', 'VBB3', 'INVERTER2_VS